<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/synth_suite_bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Synth Ecosystem Bridge: synth-pdb & synth-saxs

This tutorial demonstrates how to leverage the interoperability of the Synth suite. We will:
1. Generate a synthetic protein structure using `synth-pdb`.
2. Perform a brief physics-based structural relaxation.
3. Compute and visualize the resulting SAXS profile using `synth-saxs`.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/elkins-lab/synth-saxs.git synth-pdb biotite matplotlib
else:
    sys.path.append("../../")

In [ ]:
from synth_pdb import EnergyMinimizer, PeptideGenerator

from synth_saxs import calculate_saxs_profile, plot_saxs_results

## 1. Generating a Synthetic Protein Structure
We use `synth-pdb` to rapidly assemble a fully extended (unfolded) polypeptide from a sequence.

In [ ]:
# Define a 40-residue sequence
sequence = "ACDEFGHIKLMNPQRSTVWY"

print("Generating structure...")
generator = PeptideGenerator(sequence)
result = generator.generate(conformation="extended")
structure = result.structure

print(f"Generated structure with {len(structure)} atoms.")

## 2. Equilibrating the Structure
A fully extended chain is highly unstable in solvent. We use `synth-pdb`'s `EnergyMinimizer` to run a brief Thermal Equilibration (Molecular Dynamics at 300K). By applying a cyclic constraint (`cyclic=True`), we forcibly bring the N- and C-termini together. This rapidly drives the extended coil to fold into a highly compact globule during the MD run, creating a massive structural change.

In [ ]:
import biotite.structure.io.pdb as pdb

print("Equilibrating structure (running MD at 300K)...")
# Save the initial structure
pdb_file = pdb.PDBFile()
pdb_file.set_structure(structure)
pdb_file.write("unrelaxed.pdb")

minimizer = EnergyMinimizer()

# 1. Minimize first to resolve any severe steric clashes
print("Minimizing clashes...")
# We use cyclic=True to force the N and C termini together!
minimizer.minimize("unrelaxed.pdb", "minimized.pdb", cyclic=True)

# 2. Run thermal equilibration (10000 steps = 20 ps)
print("Running MD...")
minimizer.equilibrate("minimized.pdb", "equilibrated.pdb", steps=10000, cyclic=True)

# Load the equilibrated structure
eq_pdb = pdb.PDBFile.read("equilibrated.pdb")
eq_structure = eq_pdb.get_structure()[0]

print("Equilibration complete.")

## 3. Simulating the SAXS Profile
We will compute the SAXS profiles for both the idealized and equilibrated structures to see how the thermal compaction affects the scattering curve.

In [ ]:
# Idealized structure
q_ideal, i_ideal = calculate_saxs_profile(structure, q_min=0.01, q_max=0.3, n_points=100)

# Equilibrated structure
q_eq, i_eq = calculate_saxs_profile(eq_structure, q_min=0.01, q_max=0.3, n_points=100)

## 4. Visualizing the Results
Finally, we use `synth-saxs` to plot the standard SAXS curves.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.semilogy(q_ideal, i_ideal, "b-", linewidth=2, label=r"Extended Coil")
plt.semilogy(q_eq, i_eq, "r--", linewidth=2, label="Equilibrated Structure")
plt.xlabel(r"q ($\AA^{-1}$)", fontsize=12)
plt.ylabel("log I(q)", fontsize=12)
plt.title("Effect of Thermal Equilibration on SAXS Profile", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()

# We can also plot the full standard suite for the equilibrated structure
plot_saxs_results(q_eq, i_eq, plot_type="all")